In [1]:
import pandas as pd
from langdetect import detect, DetectorFactory
from sklearn.feature_extraction.text import TfidfVectorizer
import nltk
from nltk.corpus import stopwords

In [2]:
DetectorFactory.seed = 0

In [3]:
df = pd.read_csv('../data/data.csv')

In [4]:
df.head()

,Reviews,Label
0,De kameleonachtige uitvoering van Kurt Russell...,Positive
1,Het was een extreem laag budget (sommige scène...,Positive
2,James Cagney staat vooral bekend om zijn stoer...,Positive
3,"In navolging van het briljante ""GoyÃ´kiba"" (oo...",Positive
4,Eén van de laatste klassiekers van de Franse N...,Positive


In [5]:
df.shape

(4800, 2)

In [6]:
df['Label'].value_counts()

Label
Positive    2250
Average     2250
Negative     300
Name: count, dtype: int64

In [7]:
df['Reviews'].str.len().describe()        # char length


count    4800.000000
mean     1531.151458
std      1126.239328
min        60.000000
25%       785.000000
50%      1143.500000
75%      1929.250000
max      7654.000000
Name: Reviews, dtype: float64

In [8]:
df['Reviews'].isna().sum()                # any missing


np.int64(0)

In [9]:
df.duplicated().sum()                    # any dupes

np.int64(2)

In [10]:
df = df.drop_duplicates()


In [11]:
df.duplicated().sum()                    # any dupes

np.int64(0)

In [12]:
#df['Reviews'].apply(detect)

In [ ]:
def safe_detect(text):
    """
    Safely detects the language of a text string.
    """
    try:
        # Ensure the input is treated as a string
        return detect(str(text))
    except Exception:
        return "unknown"


In [14]:
df['language'] = df['Reviews'].apply(safe_detect)
df.head()


,Reviews,Label,language
0,De kameleonachtige uitvoering van Kurt Russell...,Positive,nl
1,Het was een extreem laag budget (sommige scène...,Positive,nl
2,James Cagney staat vooral bekend om zijn stoer...,Positive,nl
3,"In navolging van het briljante ""GoyÃ´kiba"" (oo...",Positive,nl
4,Eén van de laatste klassiekers van de Franse N...,Positive,nl


In [15]:
df_dutch = df[df['language'] == 'nl'].copy()
print(f"\nSize after filtering for Dutch: {df_dutch.shape}")


Size after filtering for Dutch: (4312, 3)


In [16]:
df_dutch['Label'].value_counts()

Label
Positive    2098
Average     1924
Negative     290
Name: count, dtype: int64

In [17]:
df_dutch.duplicated().sum()                    # any dupes

np.int64(0)

In [18]:
# 1. Download the standard list of stop words (run this once)
# nltk.download('stopwords')

# 2. Load the Dutch stop words
dutch_stop_words = stopwords.words('dutch')

In [19]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# 1. Define our features (X) and target (y)
X = df_dutch['Reviews']
y = df_dutch['Label']

# 2. Split the data (80% training, 20% testing)
# stratify=y ensures the 80/20 split maintains our imbalanced ratio

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"Training set size: {X_train.shape[0]}")
print(f"Testing set size: {X_test.shape[0]}")

# 3. Text Preprocessing: TF-IDF

vectorizer = TfidfVectorizer(max_features=5000) # Keeps the 5000 terms with the highest term frequency across the corpus
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# 4. Initialize and Train the Model
# class_weight='balanced' automatically adjusts for those 290 negative reviews
model = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)

print("\nTraining the model...")
model.fit(X_train_vec, y_train)

# 5. Make predictions and evaluate
y_pred = model.predict(X_test_vec)

print("\n--- Model Evaluation ---")
# Using a classification report gives us F1-score, which is much better than pure accuracy for imbalanced data
print(classification_report(y_test, y_pred))

Training set size: 3449
Testing set size: 863

Training the model...

--- Model Evaluation ---
              precision    recall  f1-score   support

     Average       0.61      0.63      0.62       385
    Negative       0.53      0.67      0.60        58
    Positive       0.66      0.62      0.64       420

    accuracy                           0.63       863
   macro avg       0.60      0.64      0.62       863
weighted avg       0.63      0.63      0.63       863



In [24]:
# Experiment: does removing Dutch stopwords help?
# Hypothesis: stopword lists include negation words (niet, geen, nooit)
# which carry sentiment, so removing them may hurt the Negative class.

configs = {
    "baseline (no stopwords)": TfidfVectorizer(max_features=5000),
    "with dutch stopwords": TfidfVectorizer(max_features=5000, stop_words=dutch_stop_words),
    "with bigrams": TfidfVectorizer(max_features=5000, ngram_range=(1, 2)),
}

for name, vec in configs.items():
    Xtr = vec.fit_transform(X_train)
    Xte = vec.transform(X_test)
    m = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
    m.fit(Xtr, y_train)
    pred = m.predict(Xte)
    report = classification_report(y_test, pred, output_dict=True)
    neg = report['Negative']
    macro_f1 = report['macro avg']['f1-score']
    print(f"{name:28s}  macro-F1={macro_f1:.3f}" 
          f" Neg precision={neg['precision']:.3f}  Neg recall={neg['recall']:.3f}")

baseline (no stopwords)       macro-F1=0.619 Neg precision=0.534  Neg recall=0.672
with dutch stopwords          macro-F1=0.610 Neg precision=0.596  Neg recall=0.586
with bigrams                  macro-F1=0.630 Neg precision=0.542  Neg recall=0.672


In [25]:
import numpy as np

feature_names = np.array(vectorizer.get_feature_names_out())
classes = model.classes_   # array like ['Average', 'Negative', 'Positive']

top_n = 15
for i, class_label in enumerate(classes):
    coefs = model.coef_[i]
    top_idx = np.argsort(coefs)[-top_n:][::-1]   # highest coefficients
    top_words = feature_names[top_idx]
    print(f"\nTop {top_n} words pushing toward '{class_label}':")
    print(", ".join(top_words))


Top 15 words pushing toward 'Average':
goed, beetje, leuke, hoewel, enkele, echter, leuk, behoorlijk, toch, ondanks, vond, waard, maar, nogal, vermakelijk

Top 15 words pushing toward 'Negative':
slecht, slechtste, verschrikkelijk, vreselijke, slechte, saai, enige, ergste, geld, had, geen, schrijven, regisseur, minuten, acteerwerk

Top 15 words pushing toward 'Positive':
geweldig, geweldige, beste, iedereen, prachtig, keer, show, fantastisch, uitstekende, perfect, liefde, toen, prachtige, favoriete, ben


In [36]:
def explain_prediction(review_text, vectorizer, model, top_n=10):
    """Show which words drove the prediction for a single review."""
    vec = vectorizer.transform([review_text])           # 1 x n_features, sparse
    pred_class = model.predict(vec)[0]
    class_idx = list(model.classes_).index(pred_class)
    coefs = model.coef_[class_idx]

    # contribution of each present word = its tfidf value * its coefficient
    vec_array = vec.toarray()[0]
    contributions = vec_array * coefs

    feature_names = vectorizer.get_feature_names_out()
    nonzero = np.where(vec_array > 0)[0]
    ranked = sorted(nonzero, key=lambda j: contributions[j], reverse=True)

    print(f"Predicted: {pred_class}\n")
    print(f"Top words pushing toward '{pred_class}':")
    for j in ranked[:top_n]:
        print(f"  {feature_names[j]:25s}  contribution={contributions[j]:+.3f}")

# try it
explain_prediction(X_test.iloc[860], vectorizer, model)

Predicted: Negative

Top words pushing toward 'Negative':
  oh                         contribution=+0.249
  niet                       contribution=+0.163
  slechte                    contribution=+0.155
  ik                         contribution=+0.123
  seks                       contribution=+0.117
  enige                      contribution=+0.114
  grappen                    contribution=+0.104
  idioot                     contribution=+0.103
  dit                        contribution=+0.098
  dat                        contribution=+0.095


In [37]:
# Final model config (bigrams), fit explicitly so we own these objects
final_vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_vec = final_vectorizer.fit_transform(X_train)

final_model = LogisticRegression(
    class_weight='balanced', max_iter=1000, random_state=42
)
final_model.fit(X_train_vec, y_train)

# rerun top-words with final_vectorizer and final_model
feature_names = np.array(final_vectorizer.get_feature_names_out())
classes = final_model.classes_
top_n = 15
for i, class_label in enumerate(classes):
    coefs = final_model.coef_[i]
    top_idx = np.argsort(coefs)[-top_n:][::-1]
    print(f"\nTop {top_n} for '{class_label}':")
    print(", ".join(feature_names[top_idx]))


Top 15 for 'Average':
goed, beetje, leuke, een beetje, hoewel, enkele, echter, maar het, een leuke, leuk, behoorlijk, toch, ondanks, vermakelijk, dus

Top 15 for 'Negative':
slecht, verschrikkelijk, slechtste, de slechtste, vreselijke, saai, geen, slechte, ergste, geld, had, zo slecht, enige, door een, het enige

Top 15 for 'Positive':
geweldig, geweldige, prachtig, beste, iedereen, een van, show, fantastisch, een geweldige, keer, de beste, geweldige film, liefde, perfect, uitstekende
